In [1]:

# ==========================================================
# STRESS-STRAIN ANALYSIS USING PYTHON AND PANDAS
# ==========================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
from google.colab import files

# STEP 2: Upload the CSV file

uploaded = files.upload()

# Read the uploaded CSV file
df = pd.read_csv('stress_strain_practice.csv')

# STEP 3: Display the first 5 rows

print("FIRST 5 ROWS OF THE DATASET")
print(df.head())

# STEP 4: Calculate Stress (MPa)
# Stress = Force / Cross-sectional Area
# N/mm² is equivalent to MPa

df['Stress_MPa'] = (
    df['Force_N'] / df['Cross_Section_Area_mm2']
)

# STEP 5: Calculate Strain (%)
# Strain = (Elongation / Original Length) * 100

df['Strain_Percent'] = (
    df['Elongation_mm'] / df['Original_Length_mm']
) * 100

# Display calculated results

print("\nSTRESS AND STRAIN CALCULATIONS")
print(df[['Test_ID', 'Material',
          'Stress_MPa', 'Strain_Percent']])

# STEP 6: Group results by Material
# Calculate average Stress and Strain

material_results = df.groupby('Material').agg(
    Average_Stress_MPa=('Stress_MPa', 'mean'),
    Average_Strain_Percent=('Strain_Percent', 'mean')
).reset_index()

print("\nAVERAGE STRESS AND STRAIN BY MATERIAL")
print(material_results.round(3))

# STEP 7: Identify the strongest material
# Strongest material = Highest average stress

strongest_material = material_results.loc[
    material_results['Average_Stress_MPa'].idxmax()
]

print("\nSTRONGEST MATERIAL")
print("Material:", strongest_material['Material'])
print("Average Stress:",
      round(strongest_material['Average_Stress_MPa'], 3),
      "MPa")

# STEP 8: Calculate minimum, maximum and average stress
# for each material

stress_summary = df.groupby('Material').agg(
    Minimum_Stress_MPa=('Stress_MPa', 'min'),
    Maximum_Stress_MPa=('Stress_MPa', 'max'),
    Average_Stress_MPa=('Stress_MPa', 'mean')
).reset_index()

print("\nSTRESS SUMMARY BY MATERIAL")
print(stress_summary.round(3))

# STEP 9: Export detailed results to CSV

df.to_csv('stress_strain_results.csv', index=False)

# Export material-wise average results

material_results.to_csv(
    'material_average_summary.csv', index=False
)

# Export minimum, maximum and average stress summary

stress_summary.to_csv(
    'material_stress_summary.csv', index=False
)

# STEP 10: Download the generated CSV files

print("\nCSV FILES GENERATED SUCCESSFULLY!")

files.download('stress_strain_results.csv')
files.download('material_average_summary.csv')
files.download('material_stress_summary.csv')


Saving stress_strain_practice.csv to stress_strain_practice (1).csv
FIRST 5 ROWS OF THE DATASET
  Test_ID  Material  Force_N  Cross_Section_Area_mm2  Elongation_mm  \
0    T001     Steel     1749                      27           0.51   
1    T002  Aluminum     2901                      20           0.45   
2    T003    Copper     2463                      19           0.76   
3    T004     Brass     2197                      33           0.50   
4    T005  Titanium     1312                      33           0.45   

   Original_Length_mm   Test_Date  
0                 100  2025-01-01  
1                 100  2025-01-02  
2                 100  2025-01-03  
3                 100  2025-01-04  
4                 100  2025-01-05  

STRESS AND STRAIN CALCULATIONS
   Test_ID  Material  Stress_MPa  Strain_Percent
0     T001     Steel   64.777778            0.51
1     T002  Aluminum  145.050000            0.45
2     T003    Copper  129.631579            0.76
3     T004     Brass   66.575758 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:

# ============================================================
# STRESS-STRAIN ANALYSIS WITH VISUALIZATION AND ERROR HANDLING
# ============================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

try:

    # STEP 2: Upload CSV file

    uploaded = files.upload()

    if "stress_strain_practice.csv" not in uploaded:
        raise FileNotFoundError(
            "Please upload stress_strain_practice.csv"
        )

    df = pd.read_csv("stress_strain_practice.csv")

    # STEP 3: Validate required columns

    required_columns = [
        "Test_ID", "Material", "Force_N",
        "Cross_Section_Area_mm2", "Elongation_mm",
        "Original_Length_mm", "Test_Date"
    ]

    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    print("Dataset loaded successfully!")

    # STEP 4: Display first 5 rows

    print("\nFIRST 5 ROWS:")
    display(df.head())

    # STEP 5: Handle missing values

    print("\nMISSING VALUES BEFORE CLEANING:")
    print(df.isnull().sum())

    # Convert measurement columns to numeric.
    # Invalid text values will become NaN.

    numeric_columns = [
        "Force_N",
        "Cross_Section_Area_mm2",
        "Elongation_mm",
        "Original_Length_mm"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove rows missing essential analysis values.
    # Test_Date is not essential for stress/strain calculations.

    essential_columns = [
        "Test_ID", "Material",
        "Force_N", "Cross_Section_Area_mm2",
        "Elongation_mm", "Original_Length_mm"
    ]

    original_count = len(df)

    df = df.dropna(subset=essential_columns).copy()

    # Remove invalid zero or negative area and length values

    df = df[
        (df["Cross_Section_Area_mm2"] > 0) &
        (df["Original_Length_mm"] > 0)
    ].copy()

    removed_count = original_count - len(df)

    print("\nDATA CLEANING:")
    print("Invalid or incomplete rows removed:", removed_count)
    print("Valid rows remaining:", len(df))

    if df.empty:
        raise ValueError("No valid records available for analysis.")

    # STEP 6: Calculate Stress (MPa)
    # Stress = Force / Cross-sectional Area

    df["Stress_MPa"] = (
        df["Force_N"] / df["Cross_Section_Area_mm2"]
    )

    # STEP 7: Calculate Strain (%)
    # Strain = (Elongation / Original Length) * 100

    df["Strain_Percent"] = (
        df["Elongation_mm"] / df["Original_Length_mm"]
    ) * 100

    print("\nCALCULATED STRESS AND STRAIN:")
    display(df)

    # STEP 8: Identify tests exceeding 200 MPa

    df["High_Stress"] = df["Stress_MPa"] > 200

    high_stress_tests = df[df["High_Stress"]].copy()

    print("\nTESTS WITH STRESS ABOVE 200 MPa:")

    if not high_stress_tests.empty:
        display(
            high_stress_tests.style
            .highlight_max(
                subset=["Stress_MPa"],
                color="orange"
            )
        )
    else:
        print("No tests exceeded 200 MPa.")

    # STEP 9: Group by Material
    # Calculate average stress and strain

    material_avg = df.groupby("Material").agg(
        Average_Stress=("Stress_MPa", "mean"),
        Average_Strain=("Strain_Percent", "mean")
    ).reset_index()

    print("\nAVERAGE STRESS AND STRAIN BY MATERIAL:")
    display(material_avg.round(3))

    # STEP 10: Identify strongest material

    strongest = material_avg.loc[
        material_avg["Average_Stress"].idxmax()
    ]

    print("\nSTRONGEST MATERIAL:")
    print("Material:", strongest["Material"])
    print(
        "Average Stress:",
        round(strongest["Average_Stress"], 3),
        "MPa"
    )

    # STEP 11: Minimum, Maximum and Average Stress

    stress_summary = df.groupby("Material").agg(
        Min_Stress=("Stress_MPa", "min"),
        Max_Stress=("Stress_MPa", "max"),
        Avg_Stress=("Stress_MPa", "mean")
    ).reset_index()

    print("\nSTRESS SUMMARY:")
    display(stress_summary.round(3))

    # STEP 12: Visualization - Stress vs Material
    # Bar chart displays average stress for each material

    plt.figure(figsize=(10, 6))

    bars = plt.bar(
        material_avg["Material"],
        material_avg["Average_Stress"],
        color="skyblue",
        edgecolor="black"
    )

    # Highlight strongest material

    strongest_index = material_avg[
        "Average_Stress"
    ].idxmax()

    bars[strongest_index].set_color("orange")

    plt.title("Average Stress vs Material Type",
              fontsize=15)

    plt.xlabel("Material Type", fontsize=12)
    plt.ylabel("Average Stress (MPa)", fontsize=12)

    plt.xticks(rotation=30)
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    # Display values above each bar

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.2f}",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

    # STEP 13: Export results to CSV files

    df.to_csv("stress_strain_results.csv", index=False)

    material_avg.to_csv(
        "material_average_summary.csv", index=False
    )

    stress_summary.to_csv(
        "material_stress_summary.csv", index=False
    )

    high_stress_tests.to_csv(
        "high_stress_tests.csv", index=False
    )

    print("\nALL ANALYSIS COMPLETED SUCCESSFULLY!")

    # STEP 14: Download output CSV files

    for filename in [
        "stress_strain_results.csv",
        "material_average_summary.csv",
        "material_stress_summary.csv",
        "high_stress_tests.csv"
    ]:
        files.download(filename)

except FileNotFoundError as e:
    print("FILE ERROR:", e)

except ValueError as e:
    print("DATA ERROR:", e)

except Exception as e:
    print("UNEXPECTED ERROR:", e)


Saving stress_strain_practice.csv to stress_strain_practice (2).csv
FILE ERROR: Please upload stress_strain_practice.csv
